# Simple thresholding-based segmentation of nuclei

## Input

This recipe expects an input folder containing 3D & multichannel ```.nd2``` files.

**NOTE:** single channel images are untested at the moment.

## Output

In the specified output folder (by default a subfolder of the input folder), for each ```.nd2``` file in the input, a TIFF file named ```{input file name (without ending)}_segmented.tif``` will be created. If instance segmentation is requested, it will be done via standard Watershed on the EDT of the mask.

Furthermore, a visualization of segmentations in z-maximum-projections will be created in a subfolder.

## 0) Imports and Function definitions

Run this once

In [ ]:
import json
import warnings
from functools import reduce
from itertools import product
from operator import add
from pathlib import Path

import numpy as np
from edt import edt
from nd2 import ND2File
from scipy.ndimage import gaussian_filter
from skimage.color import label2rgb
from skimage.exposure import rescale_intensity
from skimage.filters import threshold_otsu
from skimage.io import imsave
from skimage.measure import label
from skimage.morphology import (
    ball,
    binary_closing,
    remove_small_holes,
    remove_small_objects,
)
from skimage.morphology.extrema import h_maxima
from skimage.segmentation import watershed

from visualization_utils import get_segmentation_visualization
from io_helpers import imsave_nowarnings


def get_files_and_position_combos(in_files):
    return reduce(
        add,
        [
            list(product([in_file], range(get_num_positions_nd2(in_file))))
            for in_file in in_files
        ],
    )


def get_num_positions_nd2(in_file):
    """
    get the number of xy-positions / tiles in an nd2 file, will return 1 if file is just a single (potentially multichannel) stack
    """
    with ND2File(in_file) as reader:
        return reader.sizes["P"] if "P" in reader.sizes else 1


def load_single_channel_from_nd2(file_path, channel=0, position=None):

    # do a few sanity checks concerning multi-position files
    num_positions = get_num_positions_nd2(file_path)
    if num_positions > 1 and position is None:
        raise ValueError(
            f"Multiple xy positions in file {file_path}, please specify which one to load via the position parameter."
        )
    if num_positions == 1 and position not in [0, None]:
        warnings.warn(
            f"File {file_path} does not contain multiple xy positions, will load the only available position instead of specified position {position}."
        )

    with ND2File(file_path) as reader:

        if isinstance(channel, str):
            # nice OC name without whitespace
            channel_names = list(
                map(
                    lambda s: s.channel.name.strip().replace(" ", "-"),
                    reader.metadata.channels,
                )
            )

            # try to find specified channel, otherwise error and list available channels
            try:
                channel_idx = channel_names.index(channel)
            except ValueError:
                raise ValueError(
                    f"channel {channel} not found in file. available channels: {channel_names}"
                )

        else:
            channel_idx = channel

        if num_positions == 1:
            img = np.array(reader.to_dask()[:, channel_idx])
        else:
            img = np.array(reader.to_dask()[position, :, channel_idx])

        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]

    return img, pixel_size


def threshold_segmentation(
    img,
    blur_sigma,
    small_hole_size,
    small_hole_size_perplane,
    small_object_size,
    closing_radius,
    threshold_function=threshold_otsu,
):

    # blur slightly
    if blur_sigma > 0.0:
        img = gaussian_filter(img.astype(float), blur_sigma)

    # get and apply threshold
    segmented = img > threshold_function(img)

    # some morphological cleanup
    if closing_radius > 0:
        segmented = binary_closing(segmented, ball(closing_radius))

    # remove small holes per plane first
    for plane in range(segmented.shape[0]):
        segmented[plane] = remove_small_holes(
            segmented[plane], small_hole_size_perplane
        )

    segmented = remove_small_holes(segmented, small_hole_size)
    segmented = remove_small_objects(segmented, small_object_size)

    return segmented


def edt_watershed_instance_segmentation(mask, h_maxima_threshold, pixel_size):
    dt = edt(mask, anisotropy=pixel_size)
    maxima = h_maxima(dt, h_maxima_threshold)
    segmented_instance = watershed(-dt, label(maxima), mask=mask, connectivity=2)
    return segmented_instance

## 1) Set input and parameters

In [ ]:
# path containing files to visualize
in_path = '/Volumes/nn/Julia Vogtmann/Microscopy/25JV_006'


# subdirectory containing input data
# leave empty ('') if image files are directly in in_path
in_subdirectory = ''

# path to which output is saved
# default: put results in subdirectory called 'segmentation-threshold'
out_subdirectory = 'segmentation-threshold'

## parameters for segmentation
# sigma of blur to apply before thresholding
blur_sigma = 1
# size (in pixels) of small objects/small holes to discard
small_hole_size = 100
small_hole_size_perplane = 50
small_object_size = 200
# radius of binary closing applied to mask (can be slow, set to 0 to skip)
closing_radius = 0

# channel to use for segmentation. may be either interger index (e.g., 0) or the name of the channel (OC in NIS)
channel_for_segmentation = '488-CSU-W1'

# do instance segmentation?
# can be: None/False - don't do instance segmentation
# 'connected-components' - only do connected components labelling
# 'watershed' - do watershed transform on edt of mask
instance_segmentation = None
# if instance segmentation by watershed, what is the required prominence of EDT maxima to be considered as seed points
# lower values: oversegmentation, higher values: undersegmentation
h_maxima_threshold = 1.5

# whether to save a simple png visualization of segmentation results in a subfolder
save_visualization = True

# how many images to process in parallel
# NOTE: too high didn't seem to help that much, might be better on newer conda env
n_jobs_parallel = 4


## 2) Check input files

Run this to get the list of files to process and print for verification

In [ ]:
in_path = Path(in_path)

out_path = in_path / out_subdirectory

parameter_log = {
    'in_path': str(in_path),
    'in_subdirectory': in_subdirectory,
    'channel_for_segmentation': channel_for_segmentation,
    'blur_sigma': blur_sigma,
    'small_hole_size': small_hole_size,
    'small_hole_size_perplane': small_hole_size_perplane,
    'small_object_size': small_object_size,
    'instance_segmentation': instance_segmentation,
    'h_maxima_threshold': h_maxima_threshold
}

# get all nd2 files in in_path
in_files = sorted((in_path / in_subdirectory).glob('*.nd2'))

# show for verification
in_files

## 3) run segmentation

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# function to process one file
def process_single_file(in_file, position=None):

    img, pixel_size = load_single_channel_from_nd2(in_file, channel=channel_for_segmentation, position=position)
    segmented = threshold_segmentation(img, blur_sigma, small_hole_size, small_hole_size_perplane, small_object_size, closing_radius)

    if not instance_segmentation:
        pass
    elif instance_segmentation == 'watershed':
        segmented = edt_watershed_instance_segmentation(segmented, h_maxima_threshold, pixel_size)
    elif instance_segmentation == 'connected-components':
        segmented = label(segmented)
    else:
        raise ValueError(f'instance segmentation method "{instance_segmentation}" not available')

    # convert to 8 or 16 bit as necessary
    segmented = segmented.astype(np.uint16) if segmented.max() > 255 else segmented.astype(np.uint8)
    
    if save_visualization:
        visualization_projection = get_segmentation_visualization(segmented, img)
    else:
        visualization_projection = None
    return segmented, visualization_projection

# make output directories if necessary
if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)

# segment in parallel
with ThreadPoolExecutor(n_jobs_parallel) as tpe:
    futures = []
    for in_file, position_idx in get_files_and_position_combos(in_files):
        futures.append(tpe.submit(process_single_file, in_file, position_idx))
    results = []
    for f, (in_file, position_idx) in zip(futures, get_files_and_position_combos(in_files)):
        results.append(f.result())
        print(f'segmented {str(in_file)} ({position_idx}).')

with open(out_path / 'segmentation_parameters.json', 'w') as fd:
    json.dump(parameter_log, fd, indent=1) 

for (segmented, visualization_projection), (in_file, position_idx) in zip(results, get_files_and_position_combos(in_files)):
    
    num_positions = get_num_positions_nd2(in_file)

    # make filepath for output
    outfile = out_path / (in_file.stem + ('' if num_positions == 1 else f'_stack{position_idx}' ) + '_segmented.tif')
    imsave_nowarnings(str(outfile), segmented)

    if save_visualization:

        # make filepath for output
        outfile_visualization = visualization_path / (in_file.stem + ('' if num_positions == 1 else f'_stack{position_idx}' ) + '_segmented_projection.png')            
        imsave_nowarnings(str(outfile_visualization), visualization_projection)

    
